# API embedding BAAI/bge-m3 trên Kaggle

Notebook chạy model `BAAI/bge-m3` trên GPU Kaggle và mở một API công khai qua Cloudflare Tunnel.
Máy cá nhân gọi API này khi chạy `python -m src.pipeline` để embed tin đăng vào pgvector.

**Chuẩn bị (một lần):**
1. *Settings → Accelerator*: chọn **GPU T4 x2** hoặc **GPU P100**.
2. *Settings → Internet*: bật **On** (tài khoản Kaggle cần xác minh số điện thoại).
3. (Nên làm) *Add-ons → Secrets*: tạo secret `EMBEDDING_API_KEY` với một chuỗi ngẫu nhiên dài.
   Nếu không có, notebook tự sinh key mới mỗi phiên.

**Chạy:** *Run All* ở chế độ chỉnh sửa (interactive), **không** dùng *Save Version*.
Cell 4 in ra hai dòng `EMBEDDING_API_URL=...` và `EMBEDDING_API_KEY=...`; chép vào file `.env` ở máy cá nhân.
Cell cuối giữ phiên chạy; embed xong thì *Stop session* để không tốn quota GPU.

> URL tunnel thay đổi mỗi lần chạy lại notebook, nhớ cập nhật `.env`.
> Không chia sẻ notebook kèm output vì output có API key.

In [ ]:
# 1. Cài thư viện (torch có sẵn trên Kaggle) và tải cloudflared để mở tunnel công khai
!pip install -q fastapi uvicorn sentence-transformers
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

In [ ]:
# 2. Nạp model lên GPU
import secrets

import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-m3'       # phải trùng ListingEmbedder.MODEL_NAME ở máy cá nhân
MAX_SEQ_LENGTH = 1024            # mô tả tin dài nhất ~3.000 ký tự, vừa trong 1024 token
ENCODE_BATCH_SIZE = 32
MAX_TEXTS_PER_REQUEST = 256

try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret('EMBEDDING_API_KEY')
except Exception:
    API_KEY = secrets.token_urlsafe(24)   # chưa tạo secret: dùng key ngẫu nhiên cho phiên này

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = {'torch_dtype': torch.float16} if device == 'cuda' else {}
model = SentenceTransformer(MODEL_NAME, device=device, model_kwargs=dtype)
model.max_seq_length = MAX_SEQ_LENGTH
DIMENSION = model.get_sentence_embedding_dimension()
print(f'{MODEL_NAME} trên {device}, {DIMENSION} chiều')

In [ ]:
# 3. API: GET /health, POST /embed (header X-API-Key). Chạy nền trong notebook.
import threading
import time

import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title='bge-m3 embedding API')
stats = {'requests': 0, 'texts': 0}
gpu_lock = threading.Lock()       # mỗi lúc chỉ một request dùng GPU


class EmbedRequest(BaseModel):
    texts: list[str] = Field(min_length=1, max_length=MAX_TEXTS_PER_REQUEST)


@app.get('/health')
def health():
    return {'model': MODEL_NAME, 'dimension': DIMENSION, 'device': device, **stats}


@app.post('/embed')
def embed(request: EmbedRequest, x_api_key: str = Header(default='')):
    if not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail='Sai API key')
    with gpu_lock:
        vectors = model.encode(request.texts, batch_size=ENCODE_BATCH_SIZE,
                               normalize_embeddings=True, convert_to_numpy=True)
        stats['requests'] += 1
        stats['texts'] += len(request.texts)
    return {'model': MODEL_NAME, 'dimension': DIMENSION, 'embeddings': vectors.tolist()}


threading.Thread(target=uvicorn.run, daemon=True, kwargs=dict(
    app=app, host='127.0.0.1', port=8000, log_level='warning')).start()
time.sleep(3)
print('API chạy tại http://127.0.0.1:8000')

In [ ]:
# 4. Mở Cloudflare quick tunnel (không cần tài khoản), tự kiểm tra rồi in cấu hình cho .env
import re
import subprocess

import requests

tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tunnel.stdout:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break


def drain_tunnel_log():           # đọc hết log để cloudflared không bị nghẽn pipe
    for _ in tunnel.stdout:
        pass


threading.Thread(target=drain_tunnel_log, daemon=True).start()

# DNS của tunnel mới cần vài giây để hoạt động
for _ in range(30):
    try:
        requests.get(f'{public_url}/health', timeout=10).raise_for_status()
        break
    except requests.RequestException:
        time.sleep(3)
test = requests.post(f'{public_url}/embed', headers={'X-API-Key': API_KEY},
                     json={'texts': ['Căn hộ 2 phòng ngủ gần bệnh viện ở Quận 7']}, timeout=60)
test.raise_for_status()
print('Tự kiểm tra OK:', len(test.json()['embeddings'][0]), 'chiều\n')
print('Chép hai dòng sau vào file .env ở máy cá nhân:')
print(f'EMBEDDING_API_URL={public_url}')
print(f'EMBEDDING_API_KEY={API_KEY}')

In [ ]:
# 5. Giữ phiên chạy và theo dõi số request. Embed xong: Interrupt cell này rồi Stop session.
last = None
while True:
    if stats != last:
        print(time.strftime('%H:%M:%S'), stats, flush=True)
        last = dict(stats)
    time.sleep(30)